# Imports

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem

In [2]:
import rdkit
rdkit.__version__

'2022.03.3'

# Analyze ChEMBL34 for substructures found in Boceprevir

Check if compounds with a proline moiety or the alpha-ketoamide warhead have been tested against cruzain or *Trypanosoma
cruzi*.

In [3]:
cz_34 = pd.read_csv("cz_chembl34.csv", sep=';', low_memory=False)
tc_34 = pd.read_csv("tc_chembl34.csv", sep=';', low_memory=False)

In [4]:
def smiles_to_mol(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        return mol
    except:
        return

In [5]:
cz_34['mol'] = cz_34['Smiles'].apply(smiles_to_mol)

In [6]:
cz_34['mol'].isna().sum()

38

In [7]:
tc_34['mol'] = tc_34['Smiles'].apply(smiles_to_mol)

In [8]:
tc_34['mol'].isna().sum()

150

In [9]:
cz_34_notna = cz_34.dropna(subset=['mol']).copy()
cz_34_notna.shape

(33299, 48)

In [10]:
tc_34_notna = tc_34.dropna(subset=['mol']).copy()
tc_34_notna.shape

(99738, 48)

In [11]:
# Substructure search with RDKit

# Define patterns
proline_pat = Chem.MolFromSmiles("[H][C@]12CN[C@@H]([C@]1(C2(C)C)[H])C=O")
alfa_ketoamide_pat = Chem.MolFromSmiles("NCC(C(N)=O)=O")

def proline_like(mol):
    try:
        return bool(mol.GetSubstructMatch(proline_pat))
    except:
        return

def alfa_ketoamide(mol):
    try:
        return bool(mol.GetSubstructMatch(alfa_ketoamide_pat))
    except:
        return

In [12]:
cz_34_notna['proline_like'] = cz_34_notna['mol'].apply(proline_like)
tc_34_notna['proline_like'] = tc_34_notna['mol'].apply(proline_like)

In [13]:
cz_34_notna['proline_like'].value_counts()

False    33299
Name: proline_like, dtype: int64

In [14]:
tc_34_notna['proline_like'].value_counts()

False    99738
Name: proline_like, dtype: int64

In [15]:
cz_34_notna['alfa_ketoamide'] = cz_34_notna['mol'].apply(alfa_ketoamide)
tc_34_notna['alfa_ketoamide'] = tc_34_notna['mol'].apply(alfa_ketoamide)

In [16]:
cz_34_notna['alfa_ketoamide'].value_counts()

False    33299
Name: alfa_ketoamide, dtype: int64

In [17]:
tc_34_notna['alfa_ketoamide'].value_counts()

False    99738
Name: alfa_ketoamide, dtype: int64

In [18]:
# Sanity check: boceprevir
smi_bcpv = "[H][C@]12CN(C([C@H](C(C)(C)C)NC(NC(C)(C)C)=O)=O)[C@@H]([C@]1(C2(C)C)[H])C(NC(C(C(N)=O)=O)CC3CCC3)=O"
mol_bcpv = Chem.MolFromSmiles(smi_bcpv)

print(proline_like(mol_bcpv))
print(alfa_ketoamide(mol_bcpv))

True
True
